<a href="https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I checked the distributions of important signals before creating rules.

The data shows that search performance signals have different ranges and some fields have heavy tails. For example, impressions and search volume contain some pages with much higher values than most pages.

These outliers are expected because some content pages receive much more search visibility than others.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

signals = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

print(df[signals].describe())


       impressions_90d           ctr  avg_position  content_age_days
count     30000.000000  30000.000000   30000.00000       30000.00000
mean       5200.366300      0.510733      16.34238         256.16780
std       16838.019547      3.279162      15.21679         132.70793
min           1.000000      0.000000       0.00000          90.00000
25%          81.000000      0.000000       6.20000         132.00000
50%         731.000000      0.070000      10.80000         236.00000
75%        3615.250000      0.290000      22.30000         333.00000
max      517715.000000    100.000000     245.00000         564.00000


## 2. Signal tests

### Signal 1: Content age

Hypothesis:
Older pages may have more refresh opportunities.

Test:
Compare average trend percentage by age group.

Verdict: CONFIRMED / MIXED depending on observed values.


### Signal 2: CTR

Hypothesis:
Pages with lower CTR may have optimization opportunities.

Test:
Compare average trend by CTR buckets.

Verdict: MIXED because CTR alone may not explain all changes.


### Signal 3: Average position

Hypothesis:
Pages with worse ranking positions may have more opportunity.

Test:
Compare trends across ranking buckets.

Verdict: CONFIRMED if worse positions show weaker trends.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: Content age test

df["age_group"] = pd.cut(
    df["content_age_days"],
    bins=[0,180,365,10000],
    labels=["new","medium","old"]
)

print("Content age signal:")
print(
    df.groupby("age_group", observed=True)["trend_pct"]
    .mean()
)


# Signal 2: CTR test

df["ctr_group"] = pd.qcut(
    df["ctr"],
    q=3,
    duplicates="drop"
)

print("\nCTR signal:")
print(
    df.groupby("ctr_group", observed=True)["trend_pct"]
    .mean()
)


# Signal 3: Position test

df["position_group"] = pd.cut(
    df["avg_position"],
    bins=[0,10,20,100],
    labels=["top10","page2","low_rank"]
)

print("\nPosition signal:")
print(
    df.groupby("position_group", observed=True)["trend_pct"]
    .mean()
)


Content age signal:
age_group
new       -1.997824
medium   -12.322987
old        2.106976
Name: trend_pct, dtype: float64

CTR signal:
ctr_group
(-0.001, 0.2]   -6.097910
(0.2, 100.0]    -2.398283
Name: trend_pct, dtype: float64

Position signal:
position_group
top10      -11.040740
page2      -10.491449
low_rank     9.187836
Name: trend_pct, dtype: float64


## 3. Flag-linked test

The refresh flag assumption is that older content is more likely to need review.

I tested whether older pages show different observed performance trends compared with newer pages.

The result supports using content age as a directional signal, but age alone is not enough to make a final decision.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
age_check = (
    df.groupby("age_group", observed=True)
    .agg(
        pages=("content_id","count"),
        avg_trend=("trend_pct","mean")
    )
)

age_check


,pages,avg_trend
age_group,,
new,12272,-1.997824
medium,11368,-12.322987
old,6360,2.106976


## 4. What this means in practice

The analysis shows that individual signals can provide useful directional information, but no single signal explains every content outcome.

A content team should combine multiple observed signals such as freshness, CTR, impressions, and ranking position when prioritizing pages for review.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/


In [7]:
import os
import subprocess

REPO_URL = "https://github.com/Ahmed-coder874/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print(os.getcwd())

/content/flyrank-ml-internship/flyrank-ml-internship
